# Lab 04 — Missing Data, Duplicates & Type Normalization

**Week 2 · Data Engineering for LLM Pipelines** — Gamut Technology Services

Raw data is never pipeline-ready. Before a dataset can feed a join, a profile, or an
LLM fine-tuning export, it has to be made *consistent*: missing values decided on,
types coerced, and duplicate business keys collapsed to one row. This lab walks that
path end-to-end on a deliberately messy, **synthetic** slice of *Cordwell Home &
Hardware* online customers.

### Outcomes
By the end you will be able to:
1. **Quantify** missingness with `info`, `isna().mean()`, and `value_counts(dropna=False)`.
2. **Decide** per column: drop (required keys), impute (measures), or keep-null (nullable dtype).
3. **Normalize types** — parse currency strings to numeric, and parse *mixed-format* dates correctly.
4. **De-duplicate** on a business key with a deterministic, documented policy.
5. **Contract-check** the result with a `pandera` schema before export.

> **Everything here is synthetic and clearly fictional.** No real Cordwell, Lowe's,
> customer, or product data is used anywhere in this Academy.

---
#### How to work this notebook
Each exercise has a `# TODO` and a **placeholder** you replace with your solution.
Below each exercise a `check(...)` cell reports **✅ PASS / ❌ FAIL** — it *never raises*,
so a fresh notebook runs top-to-bottom with everything FAIL until you do the work.
Aim to turn each check green in order; later steps build on earlier ones.

## Setup — build the messy dataset & the `check()` helper

Seeded, self-contained, no files or internet. Run this first.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import warnings
import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)

# ── synthetic, clearly-fictional Cordwell online customers ──────────────────
rng = np.random.default_rng(123)
n = 1_500

age = rng.integers(16, 80, size=n).astype("float64")
age[rng.random(n) < 0.04] = np.nan                         # ~4% missing ages

customers = pd.DataFrame({
    "customer_id": np.arange(n),
    # ~2% missing emails (a required key -> these rows get dropped)
    "email": [f"shopper{i}@cordwell-mail.example" if rng.random() > 0.02 else None
              for i in range(n)],
    "age": age,
    # same country spelled several ways + some missing
    "country": rng.choice(["US", "U.S.A.", "usa", "SG", "DE", "BR", "IN", None],
                          size=n, p=[.25, .05, .05, .15, .15, .20, .10, .05]),
    # deliberately MIXED date formats + some missing
    "signup_date": rng.choice(["2025-01-05", "01/06/2025", "06-01-2025",
                               "2025/01/07", None],
                              size=n, p=[.25, .25, .25, .20, .05]),
    # messy money: US thousands-commas, European decimal-comma, blanks, nulls
    "spend": rng.choice(["$12,345.60", "$0.00", "$99", "1,234.50", "€45,00", "", None],
                        size=n, p=[.15, .15, .25, .20, .15, .05, .05]),
    "is_marketing_opt_in": rng.choice([True, False, None], size=n, p=[.45, .50, .05]),
})

# inject 60 duplicate business keys (same customer_id re-appears, slightly changed)
dup_ids = rng.choice(customers["customer_id"], size=60, replace=False)
dupes = customers.loc[customers["customer_id"].isin(dup_ids)].assign(
    spend="$0.00", is_marketing_opt_in=False)
customers = pd.concat([customers, dupes], ignore_index=True)

print("raw rows:", len(customers), "(1500 base + 60 injected duplicate keys)")
customers.head()

In [ ]:
# ── soft self-check: prints PASS/FAIL, never raises ─────────────────────────
_score = {"pass": 0, "fail": 0}
def check(label, predicate):
    # predicate may be a bool OR a zero-arg callable (lambda) — the callable form
    # lets the soft-catch also cover exceptions raised while computing the answer,
    # so an unfinished exercise prints FAIL instead of crashing the notebook.
    try:
        ok = bool(predicate() if callable(predicate) else predicate)
        note = ""
    except Exception as e:
        ok, note = False, f"  [error: {type(e).__name__}: {e}]"
    _score["pass" if ok else "fail"] += 1
    print(f"{'✅ PASS' if ok else '❌ FAIL'} — {label}{note}")

def score():
    t = _score["pass"] + _score["fail"]
    print(f"\n{'='*46}\n  {_score['pass']}/{t} checks passing"
          f"  ({_score['fail']} to go)\n{'='*46}")

In [ ]:
def _bad_rejected(schema, df):
    """True if the schema correctly rejects a negative spend_usd."""
    import pandas as pd
    bad = df.head(1).copy()
    bad.loc[bad.index, "spend_usd"] = -1.0
    try:
        schema.validate(bad, lazy=True)
        return False
    except Exception:
        return True

---
## Part A — Inspect & plan a missing-data strategy

You cannot clean what you have not measured. First **quantify** missingness, then
**decide** each column's fate. The rule of thumb: a column used to *join, dedupe,
partition, or sequence* is a **required key** (drop rows missing it); a column that
is a *measure or description* can be **imputed** or **kept null** with a nullable dtype.

`info()` also reveals the pandas 3.0 dtypes: text columns arrive as the new **`str`**
dtype (not `object`), and a mixed True/False/None column stays `object`.

In [ ]:
customers.info()

### A1 — Quantify missingness

Build `null_frac`: a Series of the fraction of nulls per column, **sorted descending**.
*Hint:* `.isna().mean()` then `.sort_values(...)`.

In [ ]:
# ─── Exercise A1 ───────────────────────────────────────────────
# TODO: fraction of nulls per column, sorted high -> low
null_frac = pd.Series(dtype="float64")   # placeholder — replace me
null_frac

In [ ]:
check("A1: null_frac is a Series over all columns", len(null_frac) == customers.shape[1])
check("A1: sorted descending", len(null_frac) > 0 and list(null_frac) == sorted(null_frac, reverse=True))
check("A1: customer_id has no nulls", float(null_frac.get("customer_id", 1.0)) == 0.0)
check("A1: email null-fraction ~2%", 0.01 < float(null_frac.get("email", 0)) < 0.035)

### A2 — Drop rows missing a **required key**

`customer_id` and `email` are required (we key and contact on them). Drop rows missing
either into a fresh frame `clean`, and **log the cost** so the drop is never silent.
Use `dropna(subset=[...])` — never a bare `dropna()`, which would delete any row with a
null *anywhere*.

Note the `.copy()`: under pandas 3.0 Copy-on-Write the assignment that follows is safe
either way, but an explicit copy makes intent obvious and keeps `customers` pristine.

In [ ]:
# ─── Exercise A2 ───────────────────────────────────────────────
required = ["customer_id", "email"]
# TODO: keep only rows that have BOTH required keys; copy into `clean`
clean = customers.copy()                 # placeholder — replace me
lost = len(customers) - len(clean)
print(f"dropped {lost} rows missing {required}; kept {len(clean)}")

In [ ]:
check("A2: no null customer_id in clean", clean["customer_id"].notna().all())
check("A2: no null email in clean", clean["email"].notna().all())
check("A2: dropped exactly the missing-email rows", len(clean) == 1528)
score()

---
## Part B — Impute & normalize types

Four moves: normalize a messy category, parse money to a number, parse **mixed-format**
dates, and coerce numeric/boolean dtypes. Each is a place where a naive approach *looks*
like it works but silently corrupts data — the currency and date steps especially.

### B1 — Normalize `country` (map + `fillna`)

`US`, `U.S.A.`, and `usa` are the same country. Map the variants to `"USA"`, keep the
already-clean codes, and fill the remaining nulls with `"UNKNOWN"`. Store as `country_norm`.

The idiom `.map(mapping).fillna(original).fillna("UNKNOWN")` works because `.map` returns
`NaN` for keys not in the mapping — the first `fillna` restores those untouched values,
the second handles genuine nulls.

In [ ]:
# ─── Exercise B1 ───────────────────────────────────────────────
country_map = {"U.S.A.": "USA", "usa": "USA", "US": "USA"}
# TODO: map variants -> "USA", keep clean codes, fill nulls with "UNKNOWN"
clean["country_norm"] = pd.NA            # placeholder — replace me
clean["country_norm"].value_counts(dropna=False)

In [ ]:
check("B1: country_norm has no nulls", clean["country_norm"].notna().all())
check("B1: US/U.S.A./usa collapsed into 'USA'", (clean["country_norm"] == "USA").sum() > 480)
check("B1: nulls became 'UNKNOWN'", (clean["country_norm"] == "UNKNOWN").sum() > 0)
check("B1: no stray 'usa' or 'U.S.A.' left",
      clean["country_norm"].notna().all()
      and not clean["country_norm"].isin(["usa", "U.S.A.", "US"]).any())

### B2 — Parse `spend` to a numeric `spend_usd`

The money column mixes formats, and the trap is the **comma**:

| raw | comma means | target |
|---|---|---|
| `$12,345.60` | thousands separator | `12345.60` |
| `1,234.50` | thousands separator | `1234.50` |
| `€45,00` | **decimal** separator (European) | `45.00` |
| `$99`, `$0.00` | — | `99`, `0.00` |
| `""`, `None` | — | `NaN` |

A single blanket `str.replace(',', '')` would turn `€45,00` into `4500` — a **100× error**
on ~15% of rows. The safe recipe: strip currency symbols/spaces, detect the European
decimal pattern (`^\d+,\d{1,2}$`) and swap *that* comma for a dot, **then** delete the
remaining (thousands) commas, and finally `pd.to_numeric(..., errors="coerce")`.

*(In pandas 3.0, `errors="ignore"` was removed from `to_numeric`; use `"coerce"` or `"raise"`.)*

In [ ]:
# ─── Exercise B2 ───────────────────────────────────────────────
# TODO: parse `spend` -> float `spend_usd`, handling both comma meanings.
# Steps: astype("string").str.strip() -> remove [€$ ] -> swap European
# decimal comma for a dot -> drop thousands commas -> pd.to_numeric(coerce).
clean["spend_usd"] = pd.Series(0.0, index=clean.index)    # placeholder — replace me

print("nulls before impute:", int(clean["spend_usd"].isna().sum()))
clean.loc[clean["spend"] == "€45,00", ["spend", "spend_usd"]].head(3)

In [ ]:
_row = clean.loc[clean["spend"] == "€45,00", "spend_usd"]
check("B2: '€45,00' parsed as 45.00 (NOT 4500)", (_row.round(2) == 45.00).all())
check("B2: '$12,345.60' parsed as 12345.60",
      (clean.loc[clean["spend"] == "$12,345.60", "spend_usd"].round(2) == 12345.60).all())
check("B2: '$99' parsed as 99.0", (clean.loc[clean["spend"] == "$99", "spend_usd"] == 99.0).all())
check("B2: blanks/None became NaN", int(clean["spend_usd"].isna().sum()) > 100)

### B2b — Impute missing `spend_usd` (group-wise median)

A global median ignores that a €-market customer spends differently than a $-market one.
Impute each missing value with the **median of its `country_norm` group** via
`groupby(...).transform("median")`, then backstop any still-missing (a whole group with
no data) with `0.0`.

In [ ]:
# ─── Exercise B2b ──────────────────────────────────────────────
# TODO: fill missing spend_usd with the per-country median, then 0.0
grp_median = pd.Series(np.nan, index=clean.index)   # placeholder — replace me
clean["spend_usd"] = clean["spend_usd"]             # placeholder — replace me
clean["spend_usd"].describe().round(2)

In [ ]:
check("B2b: no missing spend_usd after impute", clean["spend_usd"].notna().all())
check("B2b: all spend_usd >= 0", (clean["spend_usd"] >= 0).all())
check("B2b: median spend is 99.0", float(clean["spend_usd"].median()) == 99.0)

### B3 — Parse **mixed-format** dates ⚠️ currency-critical

`signup_date` mixes `2025-01-05`, `01/06/2025`, `06-01-2025`, and `2025/01/07`.

**The trap:** in pandas ≥ 2.0, a plain `pd.to_datetime(col, errors="coerce")` infers a
*single* format from the first non-null value and coerces everything that doesn't match to
`NaT` — silently discarding most of your dates. The fix is **`format="mixed"`**, which
parses each element independently. You will *see* the difference below.

In [ ]:
# demonstrate the trap vs the fix (both run; compare the NaT counts)
naive = pd.to_datetime(clean["signup_date"], errors="coerce")
mixed = pd.to_datetime(clean["signup_date"], format="mixed", errors="coerce", dayfirst=False)
print("NaT with naive coerce :", int(naive.isna().sum()), "<- silently dropped!")
print("NaT with format='mixed':", int(mixed.isna().sum()), "<- only genuine nulls")

In [ ]:
# ─── Exercise B3 ───────────────────────────────────────────────
# TODO: parse `signup_date` -> `signup_dt` keeping every real date (format=?)
clean["signup_dt"] = pd.Series(pd.NaT, index=clean.index, dtype="datetime64[us]")  # placeholder
clean[["signup_date", "signup_dt"]].head(6)

In [ ]:
check("B3: signup_dt is a datetime dtype", pd.api.types.is_datetime64_any_dtype(clean["signup_dt"]))
check("B3: only genuine nulls are NaT (not the naive 1000+)", int(clean["signup_dt"].isna().sum()) < 120)
check("B3: '06-01-2025' read month-first as June 1",
      (clean.loc[clean["signup_date"] == "06-01-2025", "signup_dt"].dt.month == 6).all())

### B4 — Coerce numeric & boolean dtypes

Two type fixes:
- **`age`** still has ~4% missing. Keep the nulls with the **nullable** `Float64` dtype
  (capital F) rather than forcing an imputation — missingness here may be informative.
- **`is_marketing_opt_in`** is an `object` column of `True/False/None`. Default the nulls
  to `False` (the safe marketing policy) *then* cast to `bool`.

> **Pitfall:** casting *strings* `"True"/"False"` with `.astype(bool)` makes **everything**
> `True` (every non-empty string is truthy). That is why we `fillna` real booleans here and
> would `map({"True": True, "False": False})` if the source were text.

In [ ]:
# ─── Exercise B4 ───────────────────────────────────────────────
# TODO: age -> nullable Float64 (keep nulls); opt-in -> fillna(False) then bool
clean["age"] = clean["age"]                          # placeholder — replace me
clean["is_marketing_opt_in"] = clean["is_marketing_opt_in"]   # placeholder — replace me
clean[["age", "is_marketing_opt_in"]].dtypes

In [ ]:
check("B4: age is nullable Float64", str(clean["age"].dtype) == "Float64")
check("B4: age still has some nulls (kept, not force-imputed)", clean["age"].isna().sum() > 0)
check("B4: opt-in is bool with no nulls",
      clean["is_marketing_opt_in"].dtype == bool and clean["is_marketing_opt_in"].notna().all())
score()

---
## Part C — Duplicates & de-duplication

The 60 injected duplicates share a `customer_id` (the business key) but differ on other
columns. Duplication is only meaningful **relative to a key**: `df.duplicated()` with no
`subset` compares whole rows; we care about *one row per customer*.

### C1 — Detect duplicates by business key

Build a boolean mask of every row whose `customer_id` appears more than once
(`keep=False` marks *all* members of a dup group, not just the extras).

In [ ]:
# ─── Exercise C1 ───────────────────────────────────────────────
# TODO: mark ALL rows whose customer_id is duplicated (keep=False)
dup_mask = pd.Series(False, index=clean.index)      # placeholder — replace me
print("rows in a dup group:", int(dup_mask.sum()),
      "| distinct dup keys:", clean.loc[dup_mask, "customer_id"].nunique())
clean.loc[dup_mask].sort_values("customer_id").head(4)

In [ ]:
_ref_mask = clean.duplicated(subset=["customer_id"], keep=False)   # the correct answer
check("C1: dup_mask flags all members of each dup group (keep=False)",
      pd.Series(dup_mask).reset_index(drop=True).equals(_ref_mask.reset_index(drop=True)))
check("C1: every flagged customer_id truly appears more than once",
      dup_mask.sum() > 0 and (clean.loc[dup_mask, "customer_id"].value_counts() >= 2).all())

### C2 — Resolve to one row per key (deterministic policy)

Policy: **newest signup wins; break ties by higher spend.** The key discipline is to
**sort before `drop_duplicates`** so `keep="first"` is deterministic — an unsorted
`drop_duplicates` keeps whichever row pandas happened to see first.

*(Rows with `NaT` signup sort last under `ascending=False`, so a real date always wins over
a missing one — exactly what we want.)*

In [ ]:
# ─── Exercise C2 ───────────────────────────────────────────────
# TODO: sort by customer_id, then newest signup_dt, then highest spend_usd;
#       keep the first row per customer_id.
resolved = clean                                    # placeholder — replace me
print("before:", len(clean), "-> after dedupe:", len(resolved))

In [ ]:
check("C2: one row per customer_id", resolved["customer_id"].is_unique)
check("C2: collapsed to exactly one row per unique key",
      len(resolved) == clean["customer_id"].nunique())
def _kept_newest():
    key = clean["customer_id"].value_counts().idxmax()      # a definitely-duplicated key
    kept = resolved.loc[resolved["customer_id"] == key, "signup_dt"].iloc[0]
    newest = clean.loc[clean["customer_id"] == key, "signup_dt"].max()
    return kept == newest
check("C2: kept the newest signup within a dup group", _kept_newest)

### C3 — (Alternative) column-wise reduce via `groupby().agg`

Sometimes you do not want a single surviving row but a **coalesced** one — newest email,
max spend, first non-null age. Express that per-column with a custom reducer. This is the
pattern you reach for when different columns need different survival rules.

In [ ]:
# ─── Exercise C3 ───────────────────────────────────────────────
# TODO: write first_non_null(series) -> first non-null value (or pd.NA),
# then groupby('customer_id').agg(...) coalescing columns as described above.
def first_non_null(series):
    return pd.NA                                     # placeholder — replace me

best = pd.DataFrame({"customer_id": clean["customer_id"].unique()})  # placeholder
print("coalesced rows:", len(best))
best.head(3)

In [ ]:
check("C3: one coalesced row per customer",
      lambda: len(best) == resolved.shape[0] and best["customer_id"].is_unique)
check("C3: first_non_null returns a real value when one exists",
      lambda: first_non_null(pd.Series([pd.NA, 40.0, 25.0])) == 40.0)
check("C3: spend coalesced as the group max",
      lambda: float(best["spend_usd"].sum().round(2)) == float(
          clean.groupby("customer_id")["spend_usd"].max().sum().round(2)))
score()

---
## Part D — `apply` vs vectorized (measure, don't assume)

`Series.map`/`apply` runs a Python function per element — flexible, but a full Python loop.
Vectorized string ops are usually faster **on simple, single-pass transforms**. But there is
a catch you will *see* below: each `.str.*` call scans the whole column, so a multi-step
pipeline (strip → detect → swap → drop) makes several passes, and its speed edge over a
one-pass `map` can shrink to nothing. The real rule is *measure*, then prefer the clearer
code when speed is a wash. Here you will write the parse as a per-element function, prove it
agrees with the vectorized version, then time them honestly.

### D1 — A per-element `parse_spend`

Handle: `None`/`NaN` → `None`; strip `€` and spaces; if exactly one comma and no dot, treat
the comma as a decimal; drop `$` and thousands commas; `float()`, returning `None` on failure.

In [ ]:
# ─── Exercise D1 ───────────────────────────────────────────────
def parse_spend(x: object) -> float | None:
    # TODO: implement per the spec above
    return None                                      # placeholder — replace me

apply_out = clean["spend"].map(parse_spend)
print("parse_spend('€45,00') =", parse_spend("€45,00"))
print("parse_spend('$12,345.60') =", parse_spend("$12,345.60"))

In [ ]:
# re-derive the vectorized parse from the RAW spend (pre-impute) to compare fairly
_vec = clean["spend"].astype("string").str.strip().str.replace(r"[€$\s]", "", regex=True)
_euro = _vec.str.match(r"^\d+,\d{1,2}$").fillna(False)
_vec = _vec.mask(_euro, _vec.str.replace(",", ".", regex=False)).str.replace(",", "", regex=False)
_vec = pd.to_numeric(_vec, errors="coerce")
_agree = (apply_out.astype("Float64").fillna(-1).round(2) == _vec.astype("Float64").fillna(-1).round(2))
check("D1: parse_spend('€45,00') == 45.0", parse_spend("€45,00") == 45.0)
check("D1: parse_spend('$12,345.60') == 12345.6", parse_spend("$12,345.60") == 12345.6)
check("D1: parse_spend(None) is None", parse_spend(None) is None)
check("D1: apply agrees with vectorized on every row", bool(_agree.all()))

### D2 — Time it (on a *large* frame, or you just measure noise)

At ~1.5k rows everything finishes in microseconds and the timing flips run to run. Scale to
~200k rows for a stable read. First the **string parse** — the correct multi-pass vectorized
pipeline vs the per-element `map`:

In [ ]:
big = pd.Series(np.tile(clean["spend"].to_numpy(), 130))   # ~200k rows, same value mix
print(f"timing on {len(big):,} rows\n")

def vec_full(col):                    # the CORRECT, multi-pass pipeline from B2
    s = col.astype("string").str.strip().str.replace(r"[€$\s]", "", regex=True)
    euro = s.str.match(r"^\d+,\d{1,2}$").fillna(False)
    s = s.mask(euro, s.str.replace(",", ".", regex=False)).str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

print("full correct vectorized parse (multi-pass):")
%timeit -n 3 -r 3 vec_full(big)
print("\nper-element map(parse_spend):")
%timeit -n 3 -r 3 big.map(parse_spend)

Surprised? On pandas 3.0's new **`str`** dtype these land **within a hair of each
other** — sometimes `map` even wins. The correct parse needs ~7 column scans (strip, detect,
swap, drop, convert), and that erases the usual vectorization edge. The "vectorized string ops
are always several× faster" line from older tutorials simply does not hold here.

Now the case where vectorization **does** dominate — **numeric, single-expression** work, and
especially never doing row-wise `apply(axis=1)`:

In [ ]:
_rng = np.random.default_rng(0)
num = pd.DataFrame({"q": _rng.integers(1, 6, 200_000).astype("float64"),
                    "p": _rng.choice([4.99, 12.50, 29.99, 99.00], 200_000)})
print(f"timing q*p on {len(num):,} rows\n")
print("vectorized  (num['q'] * num['p']):")
%timeit -n 3 -r 3 num["q"] * num["p"]
print("\nrow-wise  num.apply(lambda r: r['q']*r['p'], axis=1):")
%timeit -n 1 -r 2 num.apply(lambda r: r["q"] * r["p"], axis=1)

There it is: the vectorized product is **hundreds to ~1000× faster** than the
row-wise `apply`. That is the reliable, dramatic win.

**Takeaway (accurate for this stack):**
- **Numeric / single-expression work → vectorize reflexively.** Never reach for
  `apply(..., axis=1)` when a column expression exists; it is orders of magnitude slower.
- **Multi-branch *string parsing* → measure.** A clear per-element `map` can match a correct
  multi-pass vectorized pipeline on the new `str` dtype and reads better — let readability win the tie.
- Whichever you choose, **parse once and cache** the result column.

⚠️ **Currency flag:** the blanket "always vectorize, it's several× faster" advice predates the
pandas-3.0 string dtype. It holds for numeric/single-pass ops; for multi-pass string parsing,
don't quote a speedup you didn't measure.

---
## Part E — Stretch: composite-key de-duplication (self-contained)

Real de-duplication often runs on a **composite key**. Here is a tiny synthetic
*order-lines* table (generated in-notebook — no external files). Duplicate
`(order_id, product_id)` lines exist; keep the one with the largest **extended price**
(`quantity * unit_price`).

In [ ]:
rng2 = np.random.default_rng(7)
m = 400
order_lines = pd.DataFrame({
    "order_id": rng2.integers(1000, 1050, size=m),
    "product_id": rng2.integers(1, 20, size=m),
    "quantity": rng2.integers(1, 6, size=m),
    "unit_price": rng2.choice([4.99, 12.50, 29.99, 99.00], size=m),
})
# force some duplicate composite keys
d = order_lines.sample(50, random_state=3).assign(quantity=1)
order_lines = pd.concat([order_lines, d], ignore_index=True)
print("order-line rows:", len(order_lines),
      "| duplicate (order_id, product_id) rows:",
      int(order_lines.duplicated(["order_id", "product_id"], keep=False).sum()))

In [ ]:
# ─── Exercise E ────────────────────────────────────────────────
# TODO: add ext_price = quantity*unit_price; keep the max-ext_price row
#       per (order_id, product_id).
order_lines["ext_price"] = 0.0                       # placeholder — replace me
orders_dedup = order_lines                           # placeholder — replace me
print("before:", len(order_lines), "-> after:", len(orders_dedup))

In [ ]:
check("E: composite key is unique after dedupe",
      not orders_dedup.duplicated(["order_id", "product_id"]).any())
check("E: kept the max extended-price line per key",
      lambda: (not orders_dedup.duplicated(["order_id", "product_id"]).any())
      and (order_lines["ext_price"].abs().sum() > 0)
      and float(orders_dedup["ext_price"].sum().round(2)) == float(
          order_lines.groupby(["order_id", "product_id"])["ext_price"].max().sum().round(2)))

---
## Part F — Contract-check & export

Before this dataset feeds a downstream join or an LLM export, assert its shape with a
**`pandera`** schema — a data-quality gate is a *safety* gate: bad data silently poisons a
pipeline the same way bad code crashes one.

> Modern import is **`import pandera.pandas as pa`** (top-level `import pandera as pa`
> emits a `FutureWarning` in current versions). And under pandas 3.0, text columns are the
> **`str`** dtype — declare `Column(str)`, **not** `Column(object)`, or validation fails
> with *“expected object, got str.”*

### F1 — Define & validate the schema

In [ ]:
# ─── Exercise F ────────────────────────────────────────────────
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

# TODO: build a DataFrameSchema for customer_id (int, unique), email (str),
# country_norm (str), spend_usd (float >= 0), is_marketing_opt_in (bool);
# all non-nullable. Then schema.validate(resolved, lazy=True).
schema = None                                        # placeholder — replace me
validated = resolved                                 # placeholder — replace me
print("pandera: contract satisfied for", len(validated), "rows")

In [ ]:
check("F: schema is a pandera DataFrameSchema",
      lambda: schema.__class__.__name__ == "DataFrameSchema")
check("F: resolved passes the contract",
      lambda: schema.validate(resolved, lazy=True).shape[0] == resolved.shape[0])
check("F: contract rejects a negative spend",
      lambda: schema.__class__.__name__ == "DataFrameSchema" and _bad_rejected(schema, resolved))

### F2 — Export the clean dataset

Write Parquet for downstream labs (Parquet preserves dtypes; CSV would lose them).

In [ ]:
# ─── Export ────────────────────────────────────────────────────
from pathlib import Path
out = Path("artifacts"); out.mkdir(exist_ok=True)
resolved.to_parquet(out / "customers_clean.parquet", index=False)
print("wrote", (out / "customers_clean.parquet").stat().st_size, "bytes;",
      len(resolved), "rows,", resolved.shape[1], "columns")
score()

---
## Wrap-up — answer in a Markdown cell

1. **Which columns did you drop vs impute vs keep-null, and why?** Tie each choice to its
   role (key / measure / description).
2. **What de-duplication policy did you choose?** How does it treat ties and `NaT` dates?
3. **Show a before/after `dtypes` comparison** and justify two `astype`/`to_datetime` choices.

### What you practiced
Quantifying missingness → deciding per column → parsing messy money and **mixed-format
dates** without silent data loss → deterministic key de-duplication → a `pandera` contract
before export. That is the shape of every real ingestion step feeding an LLM pipeline.

**Common pitfalls (keep these in muscle memory)**
- Bare `dropna()` (no `subset=`) deletes any row with a null *anywhere* — over-drops hard.
- `str.replace(',', '')` on money silently mangles European decimals (`€45,00` → `4500`).
- Plain `to_datetime(errors="coerce")` on mixed formats → most dates become `NaT` silently;
  use `format="mixed"`.
- `.astype(bool)` on *string* `"True"/"False"` makes everything `True`.
- `drop_duplicates` without a prior deterministic `sort_values` is non-reproducible.
- `Column(object)` in pandera fails on pandas-3.0 `str` columns — use `Column(str)`.